In [13]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    
torch.backends.cudnn.benchmark = True

DEVICE: cuda
NVIDIA GeForce RTX 4060 Laptop GPU


In [15]:
IMG_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 30
NUM_CLASSES = 7

DATASET_PATH = "dataset"


In [16]:
train_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [17]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATASET_PATH, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATASET_PATH, "val"),
    transform=val_transform
)


In [18]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [19]:
class ChannelAttention(nn.Module):

    def __init__(self, in_channels, ratio=8):

        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels // ratio,
                1,
                bias=False
            ),

            nn.ReLU(),

            nn.Conv2d(
                in_channels // ratio,
                in_channels,
                1,
                bias=False
            )
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = self.fc(self.avg_pool(x))

        max_out = self.fc(self.max_pool(x))

        out = avg_out + max_out

        return self.sigmoid(out)


In [20]:
class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):

        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        x = self.conv(x)

        return self.sigmoid(x)


In [21]:

class CBAM(nn.Module):

    def __init__(self, in_channels):

        super().__init__()

        self.channel_attention = ChannelAttention(
            in_channels
        )

        self.spatial_attention = SpatialAttention()

    def forward(self, x):

        x = x * self.channel_attention(x)

        x = x * self.spatial_attention(x)

        return x


In [22]:
class EmotionNet(nn.Module):

    def __init__(self, num_classes=7):

        super().__init__()

        backbone = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.DEFAULT
        )

        self.features = backbone.features

        self.cbam = CBAM(960)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(960, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.cbam(x)

        x = self.pool(x)

        x = self.classifier(x)

        return x


In [23]:
model = EmotionNet(NUM_CLASSES)

model = model.to(device)
print(next(model.parameters()).device)

# =========================================================
# LOSS + OPTIMIZER
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.2,
    patience=2
)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\karim/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth


100.0%


cuda:0


In [24]:

scaler = torch.cuda.amp.GradScaler()

# =========================================================
# TRAIN FUNCTION
# =========================================================

def train_one_epoch():

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total

    return running_loss / len(train_loader), acc

C:\Users\karim\AppData\Local\Temp\ipykernel_20392\131995001.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [25]:
def validate():

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total

    return running_loss / len(val_loader), acc

In [26]:
best_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    scheduler.step(val_loss)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

    # SAVE BEST MODEL

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_emotion_model.pth"
        )

        print("MODEL SAVED")


C:\Users\karim\AppData\Local\Temp\ipykernel_20392\131995001.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/30
Train Loss: 1.3718 | Train Acc: 46.31%
Val Loss: 1.1427 | Val Acc: 55.30%
MODEL SAVED

Epoch 2/30
Train Loss: 1.0417 | Train Acc: 59.60%
Val Loss: 0.9636 | Val Acc: 62.88%
MODEL SAVED

Epoch 3/30
Train Loss: 0.8979 | Train Acc: 65.42%
Val Loss: 0.8565 | Val Acc: 67.51%
MODEL SAVED

Epoch 4/30
Train Loss: 0.8085 | Train Acc: 69.47%
Val Loss: 0.7683 | Val Acc: 70.96%
MODEL SAVED

Epoch 5/30
Train Loss: 0.7318 | Train Acc: 72.44%
Val Loss: 0.7170 | Val Acc: 72.68%
MODEL SAVED

Epoch 6/30
Train Loss: 0.6705 | Train Acc: 74.86%
Val Loss: 0.6887 | Val Acc: 73.69%
MODEL SAVED

Epoch 7/30
Train Loss: 0.6077 | Train Acc: 77.50%
Val Loss: 0.6663 | Val Acc: 74.95%
MODEL SAVED

Epoch 8/30
Train Loss: 0.5525 | Train Acc: 79.73%
Val Loss: 0.5941 | Val Acc: 77.55%
MODEL SAVED

Epoch 9/30
Train Loss: 0.4987 | Train Acc: 81.76%
Val Loss: 0.5797 | Val Acc: 79.02%
MODEL SAVED

Epoch 10/30
Train Loss: 0.4536 | Train Acc: 83.37%
Val Loss: 0.5461 | Val Acc: 80.40%
MODEL SAVED

Epoch 11/30
Train 

In [27]:
print("\nTRAINING FINISHED")
print(f"BEST VAL ACC: {best_acc:.2f}%")


TRAINING FINISHED
BEST VAL ACC: 86.03%
